# Federated unlearning with auditing: results notebook

Reads only the saved outputs; it trains nothing and makes no choices.

Configuration was frozen on tuning seeds 101-103 using **validation** data (`FedOSD/results_soft/sweep_decision.json`). Every number here comes from the final seeds 31-33. Write-up: `FedOSD/RESULTS_SOFT_FEDOSD.md`.

Run from `isp/notebooks/` after `run_final.py`, `audit.run_audit`, `audit.relearn` and `audit.aggregate`.

In [ ]:
import json, glob, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
ROOT = '..'
RES = os.path.join(ROOT, 'results_final')
SEEDS = [31, 32, 33]
LABELS = {'original': 'Original', 'retrain': 'Retrain (gold)', 'fedosd_hard': 'FedOSD (hard)',
          'soft_fedosd': 'Soft FedOSD (ours)', 'plain_ga': 'Plain GA',
          'posttrain_plain': 'E3: post-train, plain', 'posttrain_proj': 'E4: post-train, projected'}

## 1. Pre-registered decision (tuning seeds, validation only)

In [ ]:
dec = json.load(open(os.path.join(ROOT, 'FedOSD', 'results_soft', 'sweep_decision.json')))
print(dec['verdict'])
print('eta* =', dec['eta_star'], ' G =', {k: round(v, 2) for k, v in dec['G'].items()}, ' lambda* =', dec['lambda_star'])

## 2. Final table (seeds 31-33, test split, mean ± std)

In [ ]:
S = json.load(open(os.path.join(RES, 'final_summary.json')))['summary']
keys = {'clean_acc': 'clean acc', 'asr': 'backdoor ASR', 'forget_trig_acc': 'forget triggered acc',
        'heldout_acc': 'forget held-out acc', 'floor_test': 'worst retained client',
        'mia_attack_auc_mean': 'MIA AUC', 'distance_mean_kl_retrain_to_model': 'KL to retrain',
        'gradcam_forget_train_trig_energy_in_trigger_mean': 'CAM energy in trigger'}
fmt = lambda v: f'{v[0]:.3f} ± {v[1]:.3f}' if v[0] < 2 else f'{v[0]:.2f} ± {v[1]:.2f}'
df = pd.DataFrame({LABELS[m]: {lab: fmt(S[m][k]) for k, lab in keys.items() if k in S[m]} for m in S}).T
df

## 3. Relearning test

An attacker fine-tunes each model on 32 triggered forget-client images (plain SGD, lr 0.01). A model that truly forgot should relearn the backdoor no faster than retrain.

In [ ]:
rows = [dict(r, seed=s) for s in SEEDS if os.path.exists(p := os.path.join(RES, f'seed{s}', 'relearn.json')) for r in json.load(open(p))]
R = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in zip(axes, ['asr', 'forget_acc', 'clean_acc'], ['backdoor ASR (%)', 'forget-client clean acc (%)', 'retained clean acc (%)']):
    for m, g in R.groupby('model'):
        c = g.groupby('round')[key].agg(['mean', 'std'])
        ax.plot(c.index, c['mean'], label=LABELS.get(m, m), lw=2.5 if m in ('soft_fedosd', 'retrain') else 1.2)
        ax.fill_between(c.index, c['mean'] - c['std'], c['mean'] + c['std'], alpha=0.12)
    ax.set_title(title); ax.set_xlabel('relearning step')
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()
if os.path.exists(p := os.path.join(RES, 'relearn_overall.json')):
    display(pd.DataFrame(json.load(open(p))['per_model']).T)

## 4. Grad-CAM (one figure per seed, produced by `audit.run_audit`)

In [ ]:
for s in SEEDS:
    p = os.path.join(RES, f'seed{s}', 'audit_gradcam.png')
    if os.path.exists(p):
        display(Markdown(f'**seed {s}**')); display(Image(p))

## 5. Live check on one bundle

This reloads the exported models and recomputes predictions on a few triggered test images.

In [ ]:
import sys, importlib.util, torch
B = os.path.join(ROOT, 'bundles', 'seed31')
spec = importlib.util.spec_from_file_location('bundle_api', os.path.join(B, 'bundle_api.py'))
api = importlib.util.module_from_spec(spec); spec.loader.exec_module(api)
X, y = api.get_set('trigger_test')
idx = torch.arange(8)
with torch.no_grad():
    preds = {LABELS.get(m, m): api.load_model(m).eval()(X[idx]).argmax(1).tolist() for m in api.available_models()}
print('backdoor target label:', y[idx].tolist())
pd.DataFrame(preds, index=[f'img{i}' for i in idx.tolist()]).T